# MP12 — Linear Algebra Visualizer

Learn to track matrix dimensions, interpret dot products and norms, connect eigenvectors to singular vectors, and measure rank/error/storage tradeoffs. Prerequisites: MP02 array shapes and MP09 MSE. This lesson runs a hand-checkable matrix first, then interprets a recorded real training-crop experiment. It does not assert learner mastery.

## Vectors, matrices and transformations

A column vector x in R^n contains n real numbers. A matrix A of shape m×n maps it to y=Ax of shape m. Each output y_i=sum_j A_ij x_j is a dot product between one row and the input. For A=[[2,0],[0,1]] and x=[1,2], y=[2,2]: the first coordinate stretches by two while the second is unchanged. Dot product x·y=sum_i x_i y_i measures alignment; Euclidean norm ||x||₂=sqrt(x·x) measures length. Orthogonal transformations R satisfy RᵀR=I and preserve this norm. A 90-degree rotation [[0,-1],[1,0]] maps [1,2] to [-2,1].

For an H×W image matrix X, its Frobenius norm is sqrt(sum_ij X_ij²). MSE(X,Y)=||X-Y||_F²/(HW). HWC and CHW images are rank-three arrays, not directly the two-dimensional matrices used below: select a band explicitly. Numerical units are fixed stored intensity divided by 10000; they are not calibrated reflectance.

In [ ]:
from pathlib import Path
import sys,json
import numpy as np
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src/rssr').is_dir())
sys.path.insert(0,str(root/'src'))
from mp12_tools import approximate
A=np.diag([2.,1.]); x=np.array([1.,2.]); R=np.array([[0.,-1.],[1.,0.]])
np.testing.assert_allclose(A@x,[2,2]); np.testing.assert_allclose(R@x,[-2,1])
assert np.isclose(np.linalg.norm(R@x),np.linalg.norm(x))
print('Input',x,'stretched',A@x,'rotated',R@x)

## Eigenvectors and singular value decomposition

For a square matrix A, an eigenvector v≠0 satisfies Av=lambda v: its direction does not change, only its scale. A diagonal matrix has coordinate-axis eigenvectors. A general rectangular image needs singular value decomposition (SVD), X=U diag(s) Vᵀ. In economy form, with q=min(H,W), U is H×q, s has q nonnegative descending entries, and Vᵀ is q×W. Columns of V are eigenvectors of XᵀX with eigenvalues s²; U describes the corresponding output directions. SVD exists for rectangular real matrices, even when the matrix is singular. Repeated singular values can make individual singular vectors nonunique; the reconstruction is what matters.

Keeping the first k components gives X_k=sum_{j=1}^k s_j u_j v_jᵀ. It is an approximation of a known matrix, not an inference of missing high-resolution data. Its squared Frobenius error equals sum_{j>k}s_j². Among matrices of rank at most k, truncated SVD minimizes this error. No centering is performed: the bright mean intensity may dominate the first component.

Worked example: X=diag(3,2,1). Rank one retains diag(3,0,0), loses squared energy 2²+1²=5, and has MSE=5/9. Rank two loses energy 1; rank three reconstructs exactly. Rank zero is the all-zero approximation. The function handles each explicitly.

In [ ]:
X=np.diag([3.,2.,1.])
for k in range(4):
    Y,m=approximate(X,k)
    assert np.isclose(np.linalg.norm(X-Y)**2,m['tail_energy'])
    print(k,Y.diagonal(),m['frobenius_error'],np.mean((X-Y)**2))
assert np.isclose(np.mean((X-approximate(X,1)[0])**2),5/9)

## Storage is not always reduced

A dense float64 image stores 8HW raw bytes. The rank-k factors store 8k(H+W+1) bytes: Hk for U, k singular values, and kW for Vᵀ. This excludes metadata, compression and executable code. Compression occurs only when k<HW/(H+W+1). For 64×64, dense storage is 32768 bytes; k=16 uses 16512 bytes, while k=32 uses 33024 bytes and is already larger. A low rank can also erase roads or texture, so byte reduction alone is not a scientific success criterion.

In [ ]:
H=W=64
threshold=H*W/(H+W+1)
assert 8*16*(H+W+1)==16512
assert 8*32*(H+W+1)>8*H*W
print('Rank must be below',threshold,'to reduce raw factor bytes')

## Reproduce and inspect the real experiment

Run `python scripts/linear_experiment.py` after the pinned NAIP training imagery is available in data/raw. It selects the red band of the same 64×64 training crop used in earlier lessons. The source checksum is checked before use; no test data is evaluated. Eight ranks are measured, including zero and full rank. Each error is checked against discarded singular-value energy, and MSE must not increase with rank. All displayed reconstructions share the reference range; metric inputs are not clipped.

See [reconstruction, error and storage figure](../../reports/mp12-svd.png). The numerical record below includes acquisition date, license, checksum, units and all singular values. A full-rank reconstruction reaches numerical precision, not an exact compressed representation. Interpret whether the error reduction between k=4 and k=16 justifies the added storage; inspect spatial detail rather than relying only on MSE.

In [ ]:
record=json.loads((root/'reports/mp12-svd.json').read_text())
print(record['source']['id'],record['source']['sha256'])
for row in record['rows']:
    print(row['rank'],row['mse'],row['factor_bytes'],row['dense_bytes'])
assert record['rows'][-1]['mse']<1e-25

## Practice before reading the solutions

1. A has shape 3×2 and B has shape 2×5. What is AB's shape, and does BA exist?
2. How much energy is lost at rank one for singular values [5,4,0]?
3. Is rank-64 factor storage smaller than dense storage for a 64×64 float64 image?
4. Why is an SVD reconstruction not evidence that a model recovers real satellite detail?

## Worked solutions

1. AB is 3×5. BA would require inner dimensions 5 and 3 to match, so it is undefined.
2. Squared Frobenius error is 4²+0²=16; Frobenius error is 4. MSE additionally divides by the number of entries.
3. No: 8×64×129=66048 bytes versus 32768 dense bytes, before metadata.
4. The complete HR matrix was supplied to SVD. No unseen LR-to-HR mapping was learned or tested. This is a linear-algebra approximation exercise.

Reflect: describe an image where a low-rank representation would perform poorly, and identify what spatial structure would be lost.

In [ ]:
assert (np.ones((3,2))@np.ones((2,5))).shape==(3,5)
assert approximate(np.diag([5.,4.,0.]),1)[1]['tail_energy']==16
assert 8*64*129==66048

## References and limits

[NumPy SVD documentation](https://numpy.org/doc/stable/reference/generated/numpy.linalg.svd.html) defines economy/full shapes and return ordering. Results here use NumPy's decomposition, not a from-scratch eigensolver. One training crop cannot support a dataset-wide compression claim. Red-band grayscale is not luminance and does not replace multispectral evaluation.